# MG628 Take-Home Midterm: Analysis Notebook

**Data Analysis for Managerial Decisions · Part 2: Individual Data Analysis Case Study**

This notebook does **all of the calculations and charts** for your chosen scenario. Your job is the part managers pay analysts for: **explaining what the results mean and what the company should do.**

**How to use it**

1. Choose your scenario in the first code cell below.
2. Click **Runtime › Run all**. Everything runs in under a minute.
3. In Section 7, **set your own thresholds** (required) and re-run that section. Your probabilities will differ from your classmates’, and you must justify your choices in your report.
4. After each section, read the output and complete the **✍️ Your interpretation** prompt. These become the body of your report.
5. Copy the tables and charts you need into your report (right-click a chart › *Copy image*), and give each a figure number and an explanation in your own words.

You don't need to change any code. If you want to explore further, you may add your own cells, but you remain responsible for checking any results you add.

> The datasets are simulated for educational purposes and should be treated as sample internal company data, not actual records of the companies named.

## 1. Choose your scenario

In [ ]:
#@title Select your scenario, then choose Runtime › Run all
SCENARIO = "1 - Best Buy: Retail Sales Performance"  #@param ["1 - Best Buy: Retail Sales Performance", "2 - Apple: Customer Support Operations", "3 - FedEx: Delivery and Logistics Performance", "4 - Ford: Marketing Campaign Performance"]
SCENARIO_NUMBER = int(SCENARIO[0])
print("You selected:", SCENARIO)

## 2. Load the data

The notebook loads your dataset from the course website. If that fails (for example, offline), it asks you to upload the CSV file from Blackboard.

In [ ]:
import io, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold"})
RNG = np.random.default_rng(628)   # fixed seed so everyone's simulation results match

FILES = {1: "Retail_Sales_Mid_Term_Semester.csv", 2: "Customer_Service_Mid_Term_Semester.csv",
         3: "Logistics_Mid_Term_Semester.csv", 4: "Marketing_Campaign_Mid_Term_Semester.csv"}
FILE = FILES[SCENARIO_NUMBER]
BASE_URL = "https://raw.githubusercontent.com/monroeu/MG628/main/midterm/data/"

def load(file):
    local = os.environ.get("MG628_DATA_DIR")
    if local:
        return pd.read_csv(os.path.join(local, file))
    try:
        return pd.read_csv(BASE_URL + file)
    except Exception:
        print("Couldn't reach the course website. Please upload", file)
        from google.colab import files
        uploaded = files.upload()
        return pd.read_csv(io.BytesIO(uploaded[file]))

raw = load(FILE)
df = raw.copy()
print(f"Loaded {FILE}: {df.shape[0]} rows and {df.shape[1]} columns")
df.head()

## 3. Data description

What each column means, its type, and a first look at the values.

In [ ]:
MEANINGS = {
 1: {"TransactionID": "Unique ID of the sale", "Date": "Date of the transaction", "Product": "Product sold",
     "Quantity": "Units purchased in the transaction", "Price": "Price per unit (USD)",
     "StoreLocation": "City of the store", "TotalSale": "Quantity × Price (USD)"},
 2: {"TicketID": "Unique ID of the support ticket", "DateOpened": "Date the ticket was opened",
     "CustomerID": "Anonymous customer identifier", "IssueType": "Category of the issue",
     "ResolutionTimeHours": "Hours the ticket was worked on", "Resolved": "Whether the issue was resolved (Yes/No)"},
 3: {"ShipmentID": "Unique ID of the shipment", "Origin": "City the shipment left from", "Destination": "City it was delivered to",
     "DistanceKM": "Distance traveled (km)", "WeightKG": "Shipment weight (kg)", "DeliveryTimeDays": "Days from pickup to delivery"},
 4: {"CampaignID": "Unique ID of the campaign", "Channel": "Advertising channel", "StartDate": "Campaign start date",
     "EndDate": "Campaign end date", "BudgetUSD": "Campaign budget (USD)", "LeadsGenerated": "Potential customers generated"}}[SCENARIO_NUMBER]

desc = pd.DataFrame({
    "Meaning": [MEANINGS.get(c, "") for c in raw.columns],
    "Data type (as loaded)": [str(t) for t in raw.dtypes],
    "Kind": ["Numerical" if pd.api.types.is_numeric_dtype(raw[c]) else ("Date (stored as text)" if "Date" in c else "Categorical / identifier") for c in raw.columns],
    "Unique values": [raw[c].nunique() for c in raw.columns],
    "Example": [raw[c].iloc[0] for c in raw.columns]}, index=raw.columns)
print(f"Rows: {raw.shape[0]}   Columns: {raw.shape[1]}")
desc

## 4. Data quality checks and preparation

The notebook checks for missing values, duplicates, inconsistent types and values, and possible outliers, then creates the extra variables the analysis needs.

In [ ]:
report = []
# Missing values and duplicates
missing = raw.isna().sum()
report.append(("Missing values", f"{int(missing.sum())} in total" + ("" if missing.sum() == 0 else ": " + ", ".join(f"{c} {n}" for c, n in missing.items() if n))))
report.append(("Duplicate rows", f"{int(raw.duplicated().sum())}"))
id_col = raw.columns[0]
report.append((f"Duplicate {id_col} values", f"{int(raw[id_col].duplicated().sum())}"))

# Scenario-specific preparation
if SCENARIO_NUMBER == 1:
    df["Date"] = pd.to_datetime(df["Date"])
    report.append(("Date column", "Converted from text to a date"))
    mismatch = (abs(df["Quantity"] * df["Price"] - df["TotalSale"]) > 0.01).sum()
    report.append(("TotalSale = Quantity × Price", f"Holds on every row" if mismatch == 0 else f"{mismatch} rows don't match"))
    df["Month"] = df["Date"].dt.to_period("M").astype(str)
    report.append(("New variable: Month", "Month of each sale, for trend checks"))
    KEY_NUMERIC = ["TotalSale", "Quantity", "Price"]
elif SCENARIO_NUMBER == 2:
    df["DateOpened"] = pd.to_datetime(df["DateOpened"])
    report.append(("DateOpened column", "Converted from text to a date"))
    report.append(("Resolved values", ", ".join(sorted(df["Resolved"].unique()))))
    df["ResolvedFlag"] = (df["Resolved"] == "Yes").astype(int)
    df["Unresolved"] = 1 - df["ResolvedFlag"]
    report.append(("New variables: ResolvedFlag, Unresolved", "1/0 versions of Resolved, so percentages are easy to compute"))
    report.append(("Customers", f"{df['CustomerID'].nunique()} unique customers for {len(df)} tickets"))
    KEY_NUMERIC = ["ResolutionTimeHours"]
elif SCENARIO_NUMBER == 3:
    df["Route"] = df["Origin"] + " → " + df["Destination"]
    LATE_DAYS = 7
    df["Late"] = (df["DeliveryTimeDays"] > LATE_DAYS).astype(int)
    report.append(("New variable: Route", f"Origin → Destination ({df['Route'].nunique()} different routes)"))
    report.append(("New variable: Late", f"1 if delivery took more than {LATE_DAYS} days"))
    report.append(("Same origin and destination", f"{int((df['Origin'] == df['Destination']).sum())} shipments"))
    KEY_NUMERIC = ["DeliveryTimeDays", "DistanceKM", "WeightKG"]
else:
    df["StartDate"] = pd.to_datetime(df["StartDate"]); df["EndDate"] = pd.to_datetime(df["EndDate"])
    report.append(("StartDate, EndDate", "Converted from text to dates"))
    df["DurationDays"] = (df["EndDate"] - df["StartDate"]).dt.days
    report.append(("New variable: DurationDays", f"EndDate − StartDate; {int((df['DurationDays'] < 0).sum())} campaigns end before they start"))
    df["CostPerLead"] = df["BudgetUSD"] / df["LeadsGenerated"]
    report.append(("New variable: CostPerLead", "BudgetUSD ÷ LeadsGenerated (USD per lead)"))
    KEY_NUMERIC = ["LeadsGenerated", "BudgetUSD", "CostPerLead", "DurationDays"]

# Possible outliers: the 1.5 × IQR rule
def iqr_outliers(s):
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return s[(s < lo) | (s > hi)], lo, hi
OUTLIERS = {}
for c in KEY_NUMERIC:
    out, lo, hi = iqr_outliers(df[c])
    OUTLIERS[c] = out
    report.append((f"Possible outliers in {c}", f"{len(out)} outside {lo:,.1f} to {hi:,.1f} (1.5 × IQR rule)"))

pd.DataFrame(report, columns=["Check", "Result"]).set_index("Check")

In [ ]:
# Rows flagged as possible outliers (if any)
flagged = pd.concat([df.loc[s.index] for s in OUTLIERS.values() if len(s)]) if any(len(s) for s in OUTLIERS.values()) else pd.DataFrame()
print("Rows flagged as possible outliers:" if len(flagged) else "No values fall outside the 1.5 × IQR range for the key measures.")
flagged.drop_duplicates()

### ✍️ Your interpretation: data description and preparation

- In your own words, what does each important variable measure?
- What did the checks find (missing values, duplicates, types, consistency, outliers)? What, if anything, had to change, and why?
- Which new variables were created, and why are they useful for the business question?
- If there are possible outliers, are they errors or genuine unusual cases? Should they stay in the analysis?
- **Which check mattered most for your business decision, and what would you have done if it had failed?**

*Write your answer here (double-click this cell to edit), or in your report.*

> 

## 5. Descriptive statistics

In [ ]:
def summary(cols):
    rows = {}
    for c in cols:
        s = df[c]; m = s.mode()
        rows[c] = {"Mean": s.mean(), "Median": s.median(), "Mode": m.iloc[0] if len(m) == 1 else f"{len(m)} modes",
                   "Min": s.min(), "Max": s.max(), "Range": s.max() - s.min(), "Variance": s.var(), "Std dev": s.std(),
                   "Coef. of variation": s.std() / s.mean()}
    return pd.DataFrame(rows).T
summary(KEY_NUMERIC)

In [ ]:
# Frequencies and percentages for the categorical variables
CATS = {1: ["Product", "StoreLocation"], 2: ["IssueType", "Resolved"], 3: ["Origin", "Destination"], 4: ["Channel"]}[SCENARIO_NUMBER]
for c in CATS:
    t = df[c].value_counts().to_frame("Count"); t["Percent"] = 100 * t["Count"] / t["Count"].sum()
    display(t.style.format({"Percent": "{:.1f}%"}).set_caption(f"Frequency of {c}"))

In [ ]:
# Grouped comparisons: the core of the business question
def pct(x): return 100 * x.mean()
if SCENARIO_NUMBER == 1:
    by_product = df.groupby("Product").agg(Transactions=("TransactionID", "count"), Units=("Quantity", "sum"),
        TotalSales=("TotalSale", "sum"), AvgSale=("TotalSale", "mean"), MedianSale=("TotalSale", "median")).sort_values("TotalSales", ascending=False)
    by_product["ShareOfSales%"] = 100 * by_product["TotalSales"] / by_product["TotalSales"].sum()
    by_store = df.groupby("StoreLocation").agg(Transactions=("TransactionID", "count"), Units=("Quantity", "sum"),
        Revenue=("TotalSale", "sum"), AvgSale=("TotalSale", "mean")).sort_values("Revenue", ascending=False)
    by_store["ShareOfRevenue%"] = 100 * by_store["Revenue"] / by_store["Revenue"].sum()
    display(by_product.style.format("{:,.1f}").set_caption("Sales by product"))
    display(by_store.style.format("{:,.1f}").set_caption("Revenue by store location"))
    display(df.pivot_table(index="Product", columns="StoreLocation", values="TotalSale", aggfunc="sum", fill_value=0, margins=True, margins_name="Total").style.format("{:,.0f}").set_caption("Revenue by product and store (USD)"))
elif SCENARIO_NUMBER == 2:
    by_issue = df.groupby("IssueType").agg(Tickets=("TicketID", "count"), AvgHours=("ResolutionTimeHours", "mean"),
        MedianHours=("ResolutionTimeHours", "median"), MaxHours=("ResolutionTimeHours", "max"), Unresolved=("Unresolved", "sum"),
        UnresolvedPct=("Unresolved", pct)).sort_values("UnresolvedPct", ascending=False)
    by_issue["ShareOfTickets%"] = 100 * by_issue["Tickets"] / by_issue["Tickets"].sum()
    display(by_issue.style.format("{:,.1f}").set_caption("Performance by issue type"))
    display(df.groupby("Resolved")["ResolutionTimeHours"].describe().style.format("{:,.1f}").set_caption("Resolution hours: resolved vs unresolved tickets"))
elif SCENARIO_NUMBER == 3:
    def perf(col):
        return df.groupby(col).agg(Shipments=("ShipmentID", "count"), AvgDays=("DeliveryTimeDays", "mean"), MedianDays=("DeliveryTimeDays", "median"),
            LatePct=("Late", pct), AvgDistanceKM=("DistanceKM", "mean")).sort_values("AvgDays", ascending=False)
    display(perf("Origin").style.format("{:,.1f}").set_caption("Delivery performance by origin"))
    display(perf("Destination").style.format("{:,.1f}").set_caption("Delivery performance by destination"))
    routes = perf("Route"); routes = routes[routes["Shipments"] >= 3]
    display(routes.style.format("{:,.1f}").set_caption("Routes with at least 3 shipments, slowest first"))
else:
    by_channel = df.groupby("Channel").agg(Campaigns=("CampaignID", "count"), TotalBudget=("BudgetUSD", "sum"), TotalLeads=("LeadsGenerated", "sum"),
        AvgBudget=("BudgetUSD", "mean"), AvgLeads=("LeadsGenerated", "mean"), MedianCostPerLead=("CostPerLead", "median"), AvgDurationDays=("DurationDays", "mean"))
    by_channel["ChannelCostPerLead"] = by_channel["TotalBudget"] / by_channel["TotalLeads"]
    by_channel["ShareOfBudget%"] = 100 * by_channel["TotalBudget"] / by_channel["TotalBudget"].sum()
    by_channel["ShareOfLeads%"] = 100 * by_channel["TotalLeads"] / by_channel["TotalLeads"].sum()
    display(by_channel.sort_values("ChannelCostPerLead").style.format("{:,.1f}").set_caption("Channel performance, most cost-effective first (ChannelCostPerLead = total budget ÷ total leads)"))

### ✍️ Your interpretation: descriptive statistics

- What do the averages and medians say about typical performance? Where do mean and median differ, and what does that suggest?
- How spread out are the results (range, standard deviation, coefficient of variation)? Is performance consistent?
- Which groups (products, stores, issue types, routes, channels) stand out as strongest and weakest? By how much?
- Interpret numbers in business terms (dollars, hours, days, leads), not just as statistics.

*Write your answer here (double-click this cell to edit), or in your report.*

> 

## 6. Visual analysis

Each chart has a title and labeled axes. Use at least three in your report, each with a figure number and an explanation of what it shows and why it matters.

In [ ]:
def bar(series, title, ylabel, fmt="{:,.0f}", color="#0E8C8A"):
    s = series.sort_values(ascending=False)
    ax = s.plot(kind="bar", color=color, edgecolor="none")
    ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel(s.index.name or "")
    for p, v in zip(ax.patches, s.values): ax.annotate(fmt.format(v), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()
def hist(col, title, xlabel, bins=12):
    ax = df[col].plot(kind="hist", bins=bins, color="#5B6C8F", edgecolor="white")
    ax.axvline(df[col].mean(), color="#E8A33D", lw=2, label=f"Mean {df[col].mean():,.1f}")
    ax.axvline(df[col].median(), color="#D1495B", lw=2, ls="--", label=f"Median {df[col].median():,.1f}")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Number of records"); ax.legend(); plt.tight_layout(); plt.show()
def box(col, by, title, ylabel):
    groups = df.groupby(by)[col].median().sort_values().index
    ax = plt.gca(); data = [df.loc[df[by] == g, col] for g in groups]
    style = dict(patch_artist=True, boxprops=dict(facecolor="#CFE7E6"), medianprops=dict(color="#D1495B", lw=2))
    try: ax.boxplot(data, tick_labels=list(groups), **style)
    except TypeError: ax.boxplot(data, labels=list(groups), **style)   # older matplotlib
    ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel(by); plt.tight_layout(); plt.show()
def scatter(x, y, title, xlabel, ylabel):
    r = df[x].corr(df[y]); b, a = np.polyfit(df[x], df[y], 1)
    ax = plt.gca(); ax.scatter(df[x], df[y], color="#0E8C8A", alpha=.75)
    xs = np.linspace(df[x].min(), df[x].max(), 50); ax.plot(xs, a + b * xs, color="#E8A33D", lw=2, label=f"Trend line (r = {r:.2f})")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.legend(); plt.tight_layout(); plt.show()
    return r

In [ ]:
if SCENARIO_NUMBER == 1:
    bar(by_product["TotalSales"], "Figure: Total sales by product", "Total sales (USD)")
    bar(by_store["Revenue"], "Figure: Revenue by store location", "Revenue (USD)", color="#5B6C8F")
    bar(by_product["Units"], "Figure: Units sold by product", "Units sold", color="#E8A33D")
    hist("TotalSale", "Figure: Distribution of transaction values", "Transaction value (USD)")
    box("TotalSale", "Product", "Figure: Transaction values by product", "Transaction value (USD)")
    ax = plt.gca(); ax.scatter(by_product["AvgSale"], by_product["TotalSales"], s=120, color="#0E8C8A")
    for name, rw in by_product.iterrows(): ax.annotate(name, (rw["AvgSale"], rw["TotalSales"]), xytext=(6, 4), textcoords="offset points")
    ax.set_title("Figure: Average sale vs total sales by product"); ax.set_xlabel("Average transaction value (USD)"); ax.set_ylabel("Total sales (USD)"); plt.tight_layout(); plt.show()
elif SCENARIO_NUMBER == 2:
    bar(by_issue["Tickets"], "Figure: Tickets by issue type", "Number of tickets")
    bar(by_issue["UnresolvedPct"], "Figure: Percentage of tickets unresolved, by issue type", "Unresolved (%)", fmt="{:.0f}%", color="#D1495B")
    bar(by_issue["AvgHours"], "Figure: Average resolution time by issue type", "Average hours", fmt="{:.1f}", color="#5B6C8F")
    hist("ResolutionTimeHours", "Figure: Distribution of resolution times", "Resolution time (hours)")
    box("ResolutionTimeHours", "IssueType", "Figure: Resolution time by issue type", "Hours")
    ct = pd.crosstab(df["IssueType"], df["Resolved"]); ax = ct.plot(kind="bar", stacked=True, color=["#D1495B", "#0E8C8A"])
    ax.set_title("Figure: Resolved and unresolved tickets by issue type"); ax.set_ylabel("Number of tickets"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
elif SCENARIO_NUMBER == 3:
    bar(perf("Origin")["AvgDays"], "Figure: Average delivery time by origin", "Average days", fmt="{:.1f}")
    bar(perf("Destination")["LatePct"], f"Figure: Shipments taking more than {LATE_DAYS} days, by destination", "Late shipments (%)", fmt="{:.0f}%", color="#D1495B")
    hist("DeliveryTimeDays", "Figure: Distribution of delivery times", "Delivery time (days)", bins=10)
    box("DeliveryTimeDays", "Origin", "Figure: Delivery time by origin", "Days")
    R_DISTANCE = scatter("DistanceKM", "DeliveryTimeDays", "Figure: Delivery time vs distance", "Distance (km)", "Delivery time (days)")
    R_WEIGHT = scatter("WeightKG", "DeliveryTimeDays", "Figure: Delivery time vs weight", "Weight (kg)", "Delivery time (days)")
else:
    bar(by_channel["TotalLeads"], "Figure: Total leads by channel", "Leads generated")
    bar(by_channel["ChannelCostPerLead"], "Figure: Cost per lead by channel", "USD per lead", fmt="${:,.0f}", color="#D1495B")
    bar(by_channel["TotalBudget"], "Figure: Total budget by channel", "Budget (USD)", color="#5B6C8F")
    box("LeadsGenerated", "Channel", "Figure: Leads per campaign by channel", "Leads generated")
    R_BUDGET = scatter("BudgetUSD", "LeadsGenerated", "Figure: Leads vs campaign budget", "Budget (USD)", "Leads generated")
    R_DURATION = scatter("DurationDays", "LeadsGenerated", "Figure: Leads vs campaign duration", "Duration (days)", "Leads generated")

In [ ]:
# Correlations between the numerical variables (r ranges from −1 to +1; values near 0 mean little linear relationship)
num = df.select_dtypes("number")
num = num[[c for c in num.columns if c not in ("ResolvedFlag", "Unresolved", "Late")]]
if num.shape[1] > 1:
    display(num.corr().style.format("{:.2f}").background_gradient(cmap="RdBu", vmin=-1, vmax=1).set_caption("Correlation matrix"))
    print("Rule of thumb: |r| < 0.3 weak, 0.3–0.7 moderate, > 0.7 strong. Correlation is not causation.")
else:
    print("Only one numerical measure in this dataset; relationships are explored through the group comparisons above.")

### ✍️ Your interpretation: visual analysis

Choose **exactly three** charts for your report, and explain **why you chose them over the others**. For each one:
- What does it show?
- What pattern, difference or unusual value should a manager notice?
- How does it support (or complicate) your recommendation?

If a relationship looks weak, say so. A weak or absent relationship is a valid finding; explain what it means for the decision.

*Write your answer here (double-click this cell to edit), or in your report.*

> 

## 7. Probability and risk assessment

Probabilities here are estimated from the dataset (relative frequencies).

**Required: set your own thresholds.** Change the threshold for your scenario in the next cell to a value that makes business sense to you (for example a delivery target your customers would accept, or the fewest leads a campaign must generate to be worth its budget), then run the cell and the two after it. Record your threshold and your reason; your report must explain it.

In [ ]:
#@title Thresholds (optional: change and re-run this cell)
HIGH_VALUE_USD = 2000       #@param {type:"number"}
LOW_VALUE_USD = 500         #@param {type:"number"}
HOURS_THRESHOLD = 48        #@param {type:"number"}
DELIVERY_TARGET_DAYS = 5    #@param {type:"number"}
LOW_LEADS = 250             #@param {type:"number"}
CPL_THRESHOLD_USD = 100     #@param {type:"number"}

def p(mask): return mask.mean()
rows = []
if SCENARIO_NUMBER == 1:
    rows += [(f"P(transaction ≥ ${HIGH_VALUE_USD:,})", p(df["TotalSale"] >= HIGH_VALUE_USD)),
             (f"P(transaction < ${LOW_VALUE_USD:,})", p(df["TotalSale"] < LOW_VALUE_USD))]
    for g, d in df.groupby("StoreLocation"): rows.append((f"P(transaction < ${LOW_VALUE_USD:,} | {g})", p(d["TotalSale"] < LOW_VALUE_USD)))
    for g, d in df.groupby("Product"): rows.append((f"P(transaction ≥ ${HIGH_VALUE_USD:,} | {g})", p(d["TotalSale"] >= HIGH_VALUE_USD)))
elif SCENARIO_NUMBER == 2:
    rows += [("P(ticket unresolved)", p(df["Resolved"] == "No")),
             (f"P(resolution time > {HOURS_THRESHOLD} hours)", p(df["ResolutionTimeHours"] > HOURS_THRESHOLD)),
             (f"P(unresolved AND > {HOURS_THRESHOLD} hours)", p((df["Resolved"] == "No") & (df["ResolutionTimeHours"] > HOURS_THRESHOLD)))]
    for g, d in df.groupby("IssueType"): rows.append((f"P(unresolved | {g})", p(d["Resolved"] == "No")))
    for g, d in df.groupby("IssueType"): rows.append((f"P(> {HOURS_THRESHOLD} hours | {g})", p(d["ResolutionTimeHours"] > HOURS_THRESHOLD)))
elif SCENARIO_NUMBER == 3:
    rows += [(f"P(delivery > {LATE_DAYS} days)", p(df["DeliveryTimeDays"] > LATE_DAYS)),
             (f"P(delivered within {DELIVERY_TARGET_DAYS} days)", p(df["DeliveryTimeDays"] <= DELIVERY_TARGET_DAYS))]
    for g, d in df.groupby("Origin"): rows.append((f"P(late | origin {g})", p(d["Late"] == 1)))
    for g, d in df.groupby("Destination"): rows.append((f"P(late | destination {g})", p(d["Late"] == 1)))
else:
    rows += [(f"P(campaign generates < {LOW_LEADS} leads)", p(df["LeadsGenerated"] < LOW_LEADS)),
             (f"P(cost per lead > ${CPL_THRESHOLD_USD})", p(df["CostPerLead"] > CPL_THRESHOLD_USD))]
    for g, d in df.groupby("Channel"): rows.append((f"P(< {LOW_LEADS} leads | {g})", p(d["LeadsGenerated"] < LOW_LEADS)))
    for g, d in df.groupby("Channel"): rows.append((f"P(cost per lead > ${CPL_THRESHOLD_USD} | {g})", p(d["CostPerLead"] > CPL_THRESHOLD_USD)))
PROBS = pd.DataFrame(rows, columns=["Event", "Probability"]).set_index("Event")
PROBS["As a percentage"] = (100 * PROBS["Probability"]).map("{:.1f}%".format)
PROBS["Probability"] = PROBS["Probability"].map("{:.3f}".format)
PROBS

In [ ]:
# Probability tree: first branch = group, second branch = the risk event (all probabilities from the data)
if SCENARIO_NUMBER == 1:   GROUP, EVENT, LABEL = "StoreLocation", df["TotalSale"] < LOW_VALUE_USD, f"sale < ${LOW_VALUE_USD:,}"
elif SCENARIO_NUMBER == 2: GROUP, EVENT, LABEL = "IssueType", df["Resolved"] == "No", "unresolved"
elif SCENARIO_NUMBER == 3: GROUP, EVENT, LABEL = "Origin", df["Late"] == 1, f"late (> {LATE_DAYS} days)"
else:                      GROUP, EVENT, LABEL = "Channel", df["LeadsGenerated"] < LOW_LEADS, f"< {LOW_LEADS} leads"
tree = []
for g, idx in df.groupby(GROUP).groups.items():
    pg = len(idx) / len(df); pe = EVENT.loc[idx].mean()
    tree.append({GROUP: g, f"P({GROUP})": pg, f"P({LABEL} | {GROUP})": pe, f"P({GROUP} AND {LABEL})": pg * pe})
tree = pd.DataFrame(tree).set_index(GROUP)
display(tree.style.format("{:.3f}").set_caption("Probability tree, written as a table"))
print(f"Overall P({LABEL}) = sum of the last column = {tree.iloc[:, -1].sum():.3f}")
share = tree.iloc[:, -1] / tree.iloc[:, -1].sum()
print(f"Share of all '{LABEL}' cases coming from each {GROUP}:")
print((100 * share).round(1).astype(str).add("%").to_string())

In [ ]:
# Simple Monte Carlo simulation: what could happen next, if the future looks like this data?
N_SIM = 10_000
if SCENARIO_NUMBER == 1:
    n, target = 100, df["TotalSale"].sum() * 0.9
    sims = RNG.choice(df["TotalSale"].values, size=(N_SIM, n), replace=True).sum(axis=1)
    question = f"total revenue from the next {n} transactions falls below ${target:,.0f} (90% of the current total)"
    risk = (sims < target).mean(); xlabel = f"Revenue from {n} transactions (USD)"
elif SCENARIO_NUMBER == 2:
    n = 50; k = 30; rate = (df["Resolved"] == "No").mean()
    sims = RNG.binomial(n, rate, size=N_SIM)
    question = f"at least {k} of the next {n} tickets go unresolved"; risk = (sims >= k).mean(); xlabel = f"Unresolved tickets out of {n}"
elif SCENARIO_NUMBER == 3:
    n = 30; k = 12; rate = df["Late"].mean()
    sims = RNG.binomial(n, rate, size=N_SIM)
    question = f"at least {k} of the next {n} shipments take more than {LATE_DAYS} days"; risk = (sims >= k).mean(); xlabel = f"Late shipments out of {n}"
else:
    n = 10; k = 3; rate = (df["LeadsGenerated"] < LOW_LEADS).mean()
    sims = RNG.binomial(n, rate, size=N_SIM)
    question = f"at least {k} of the next {n} campaigns generate fewer than {LOW_LEADS} leads"; risk = (sims >= k).mean(); xlabel = f"Low-lead campaigns out of {n}"
ax = plt.gca(); bins = 30 if SCENARIO_NUMBER == 1 else np.arange(sims.min() - 0.5, sims.max() + 1.5)
ax.hist(sims, bins=bins, color="#5B6C8F", edgecolor="white")
ax.set_title(f"Figure: {N_SIM:,} simulated outcomes"); ax.set_xlabel(xlabel); ax.set_ylabel("Number of simulations"); plt.tight_layout(); plt.show()
print(f"Estimated probability that {question}: {risk:.1%}")
print("Assumption: future cases behave like the ones in this dataset (same mix and rates).")

### ✍️ Your interpretation: probability and risk

- **Which threshold did you choose, and why is it the right business standard for this company?**
- Which event or risk matters most for this decision, and why?
- How was its probability estimated (relative frequency, conditional probability, probability tree or simulation)?
- What does the number mean for the company in practical terms? Which groups carry the most risk?
- What assumptions sit behind the estimate, and how could they be wrong?

*Write your answer here (double-click this cell to edit), or in your report.*

> 

## 8. Key numbers for your report

In [ ]:
key = []
if SCENARIO_NUMBER == 1:
    key += [("Total revenue", f"${df['TotalSale'].sum():,.0f}"), ("Average transaction", f"${df['TotalSale'].mean():,.0f}"),
            ("Median transaction", f"${df['TotalSale'].median():,.0f}"),
            ("Top product by sales", f"{by_product.index[0]} (${by_product['TotalSales'].iloc[0]:,.0f}, {by_product['ShareOfSales%'].iloc[0]:.1f}%)"),
            ("Lowest product by sales", f"{by_product.index[-1]} (${by_product['TotalSales'].iloc[-1]:,.0f})"),
            ("Most units sold", f"{by_product['Units'].idxmax()} ({by_product['Units'].max():,} units)"),
            ("Top store by revenue", f"{by_store.index[0]} (${by_store['Revenue'].iloc[0]:,.0f})"),
            ("Lowest store by revenue", f"{by_store.index[-1]} (${by_store['Revenue'].iloc[-1]:,.0f})")]
elif SCENARIO_NUMBER == 2:
    key += [("Tickets", f"{len(df)}"), ("Resolved", f"{df['ResolvedFlag'].mean():.1%}"), ("Unresolved", f"{df['Unresolved'].mean():.1%}"),
            ("Average resolution time", f"{df['ResolutionTimeHours'].mean():.1f} hours"), ("Median resolution time", f"{df['ResolutionTimeHours'].median():.1f} hours"),
            ("Most frequent issue", f"{df['IssueType'].value_counts().idxmax()} ({df['IssueType'].value_counts().max()} tickets)"),
            ("Highest unresolved rate", f"{by_issue.index[0]} ({by_issue['UnresolvedPct'].iloc[0]:.0f}%)"),
            ("Longest average resolution", f"{by_issue['AvgHours'].idxmax()} ({by_issue['AvgHours'].max():.1f} hours)")]
elif SCENARIO_NUMBER == 3:
    o, d = perf("Origin"), perf("Destination")
    key += [("Shipments", f"{len(df)}"), ("Average delivery time", f"{df['DeliveryTimeDays'].mean():.1f} days"), ("Median delivery time", f"{df['DeliveryTimeDays'].median():.1f} days"),
            (f"Late (> {LATE_DAYS} days)", f"{df['Late'].mean():.1%}"), ("Slowest origin", f"{o.index[0]} ({o['AvgDays'].iloc[0]:.1f} days)"),
            ("Slowest destination", f"{d.index[0]} ({d['AvgDays'].iloc[0]:.1f} days)"),
            ("Correlation: distance vs delivery time", f"r = {R_DISTANCE:.2f}"), ("Correlation: weight vs delivery time", f"r = {R_WEIGHT:.2f}")]
else:
    bc = by_channel.sort_values("ChannelCostPerLead")
    key += [("Campaigns", f"{len(df)}"), ("Total budget", f"${df['BudgetUSD'].sum():,.0f}"), ("Total leads", f"{df['LeadsGenerated'].sum():,}"),
            ("Overall cost per lead", f"${df['BudgetUSD'].sum() / df['LeadsGenerated'].sum():,.2f}"),
            ("Most cost-effective channel", f"{bc.index[0]} (${bc['ChannelCostPerLead'].iloc[0]:,.2f} per lead)"),
            ("Least cost-effective channel", f"{bc.index[-1]} (${bc['ChannelCostPerLead'].iloc[-1]:,.2f} per lead)"),
            ("Most leads in total", f"{by_channel['TotalLeads'].idxmax()} ({by_channel['TotalLeads'].max():,})"),
            ("Correlation: budget vs leads", f"r = {R_BUDGET:.2f}"), ("Correlation: duration vs leads", f"r = {R_DURATION:.2f}")]
pd.DataFrame(key, columns=["Measure", "Value"]).set_index("Measure")

### ✍️ Your interpretation: findings and recommendation

- **Findings:** What are the three to five most important results? Which differences between groups matter most? What are the strengths, weaknesses and risks?
- **Limitations:** What can't this data tell you (sample size, time period, missing variables, correlation vs causation)?
- **Recommendation:** What specific action should management take? Why is it better than the alternatives? Which numbers and figures support it? What risks remain, and what additional data should the company collect?

*"Continue analyzing the data" is not a recommendation. Recommend a direction.*

*Write your answer here (double-click this cell to edit), or in your report.*

> 

---
### Before you submit

- Every table or chart in your report has a number, a title, and an explanation in your own words.
- Your interpretations explain what the results **mean for the decision**, not just what they are.
- If you used AI tools, acknowledge it briefly in your report, as the assignment requires.